## ಎम्बೆಡ್ಡಿಂಗ್ಸ್

ನಮ್ಮ ಹಿಂದಿನ ಉದಾಹರಣೆಯಲ್ಲಿ, ನಾವು ಉದ್ದವು `vocab_size` ಇರುವ ಉನ್ನತ-ಆಯಾಮದ ಬ್ಯಾಗ್-ಆಫ್-ವರ್ಡ್ಸ್ ವೆಕ್ಟರ್‌ಗಳ ಮೇಲೆ ಕಾರ್ಯನಿರ್ವಹಿಸಿದ್ದೇವೆ, ಮತ್ತು ನಾವು ಸ್ಪಷ್ಟವಾಗಿ ಕಡಿಮೆ-ಆಯಾಮದ ಸ್ಥಾನಾತ್ಮಕ ಪ್ರತಿನಿಧಾನ ವೆಕ್ಟರ್‌ಗಳಿಂದ ವಿರಳ ಒನ್-ಹಾಟ್ ಪ್ರತಿನಿಧಾನಕ್ಕೆ ಪರಿವರ್ತನೆ ಮಾಡುತ್ತಿದ್ದುದು. ಈ ಒನ್-ಹಾಟ್ ಪ್ರತಿನಿಧಾನ ಸ್ಮೃತಿ-ಕಾರ್ಯಕ್ಷಮವಲ್ಲ, ಹೆಚ್ಚುವರಿಯಾಗಿ, ಪ್ರತಿ ಪದವನ್ನು ಪರಸ್ಪರ ಸ್ವತಂತ್ರವಾಗಿ ಪರಿಗಣಿಸಲಾಗುತ್ತದೆ, ಅಂದರೆ ಒನ್-ಹಾಟ್ ಎನ್ಕೋಡ್ಡ್ ವೆಕ್ಟರ್‌ಗಳು ಪದಗಳ ಮಧ್ಯೆ ಯಾವುದೇ ಅರ್ಥಾತ್ಮಕ ಸಮಾನತೆಯನ್ನು ವ್ಯಕ್ತಪಡಿಸುವುದಿಲ್ಲ.

ಈ ಘಟಕದಲ್ಲಿ, ನಾವು **News AG** ಡೇಟಾಸೆಟ್ ಅನ್ನು ಮುಂದುವರೆಸಿಕೊಂಡು ಪರಿಶೀಲಿಸುವೆವು. ಪ್ರಾರಂಭಿಸಲು, ಡೇಟಾವನ್ನು ಲೋಡ್ ಮಾಡೋಣ ಮತ್ತು ಹಿಂದಿನ ನೋಟ್‌ಬುಕ್‌ನಿಂದ ಕೆಲವು ವ್ಯಾಖ್ಯಾನಗಳನ್ನು ಪಡೆಯೋಣ.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## ಹೆಚ್ಚಿಸುವುದು (embedding) ಎಂದರೆ ಏನು?

**ಹೆಚ್ಚಿಸುವುದು** ಎಂಬ ಕಲ್ಪನೆ ಪದಗಳನ್ನು ಕಡಿಮೆ ಆಯಾಮದ ಸಾಂದ್ರ ವೆಕ್ಟರ್‌ಗಳ ಮೂಲಕ ಪ್ರತಿನಿಧಿಸುವದು, ಇದು ಯಾವೋ ರೀತಿಯಲ್ಲಿ ಪದದ ಅರ್ಥತಃ ಅರ್ಥವನ್ನು ಪ್ರತಿಬಿಂಬಿಸುತ್ತದೆ. ನಾವು ಮುಂದಿಗೆ ಅರ್ಥಪೂರ್ಣ ಪದ ಹೆಚ್ಚಿಸುವುದನ್ನು ಹೇಗೆ ನಿರ್ಮಿಸಲು ಎಂಬುದನ್ನು ಚರ್ಚಿಸುವೆವು, ಆದರೆ ಈಗಾಗಲೇ ಹೆಚ್ಚಿಸುವುದನ್ನು ಒಂದು ಪದ ವೆಕ್ಟರಿನ ಆಯಾಮವನ್ನು ಕಡಿಮೆ ಮಾಡುವ ರೀತಿಯಾಗಿ ಪರಿಗಣಿಸೋಣ.

ಆದ್ದರಿಂದ, ಹೆಚ್ಚಿಸುವ ರೀತಿಯ ಪದವು ಸೇರಿಸುವ ಪದವನ್ನು ಇನ್‌ಪುಟ್ ಆಗಿ ತೆಗೆದುಕೊಂಡು ನಿರ್ದಿಷ್ಟ `embedding_size` ಆಯಾಮದ ಔಟ್‌ಪುಟ್ ವೆಕ್ಟರನ್ನು ಉತ್ಪಾದಿಸುತ್ತದೆ. ಅರ್ಥದಲ್ಲಿ, ಇದು `Linear` ಪದರದೊಂದಿಗೆ ಬಹಳ ಸಮಾನವಾಗಿದೆ, ಆದರೆ ಒನ್-ಹಾಟ್ ಎನ್‌ಕೋಡ್ ಮಾಡಿದ ವೆಕ್ಟರಿನ ಬದಲಾಗಿ, ಇದು ಪದ ಸಂಖ್ಯೆ ಇನ್‌ಪುಟ್ ಆಗಿ ತೆಗೆದುಕೊಳ್ಳಬಹುದು.

ನಮ್ಮ ಜಾಲಕದಲ್ಲಿ ಮೊದಲನೆಯ ಪದರವಾಗಿ ಹೆಚ್ಚಿಸುವ ಪದರವನ್ನು ಬಳಸಿ, ನಾವು ಬ್ಯಾಗ್-ಆಫ್-ವರ್ಡ್ಸ್ ಮಾದರಿಯಿಂದ **embedding bag** ಮಾದರಿಗೆ ಮಾರ್ಗಮಾಡಬಹುದು, ಇದರಲ್ಲಿ ಮೊದಲು ನಮ್ಮ ಪಠ್ಯದ ಪ್ರತಿಯೊಂದು ಪದವನ್ನು ಅನುಸಾರವಾಗಿ ಹೆಚ್ಚಿಸುವುದಕ್ಕೆ ಪರಿವರ್ತಿಸಿ ನಂತರ ಅವನ್ನು ಎಲ್ಲಾ ಹೆಚ್ಚಿಸುವುದರ ಮೇಲೆ ಒಂದು ಏಕಮತ ಕ್ರಿಯೆಯನ್ನು ಲೆಕ್ಕಿಸಬಹುದು, ಉದಾಹರಣೆಗೆ `sum`, `average` ಅಥವಾ `max`.

![ಐದು ಸರಣಿಯ ಪದಗಳಿಗೆ ಹೆಚ್ಚಿಸುವ ವರ್ಗೀಕರಣದ ಚಿತ್ರ ಪ್ರದರ್ಶಿಸಲಾಗುತ್ತಿದೆ.](../../../../../translated_images/kn/embedding-classifier-example.b77f021a7ee67eee.webp)

ನಮ್ಮ ವರ್ಗೀಕರಣ ನ್ಯೂರಲ್ ನೆಟ್‌ವರ್ಕ್ ಆರಂಭವಾಗಿ ಹೆಚ್ಚಿಸುವ ಪದರ, ನಂತರ ಏಕತಾರವಾದ ಪದರ ಮತ್ತು ಅದರ ಮೇಲಿನ ರೇಖೀಯ ವರ್ಗೀಕರಣಪತ್ರವನ್ನು ಹೊಂದಿರುತ್ತದೆ:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### ಚರ ಬದಲಾಗುವ ಸರಣಿ ಗಾತ್ರವನ್ನು ನಿರ್ವಹಿಸುವುದು

ಈ ವಾಸ್ತುಶಿಲ್ಪದ ಫಲವಾಗಿ, ನಮ್ಮ ಜಾಲಕ್ಕೆ ಮಿನಿಬ್ಯಾಚ್‌ಗಳನ್ನು ನಿರ್ದಿಷ್ಟ ರೀತಿಯಲ್ಲಿ ರಚಿಸಬೇಕಾಗುತ್ತದೆ. ಹಿಂದಿನ ಘಟಕದಲ್ಲಿ, ಬ್ಯಾಗ್-ಆಫ್-ವರ್ಡ್ಸ್ ಬಳಸುವಾಗ, ಮಿನಿಬ್ಯಾಚ್中的 ಎಲ್ಲಾ BoW ಟೆನ್ಸರ್‌ಗಳು ಸಮಾನ ಗಾತ್ರದಲ್ಲಿದ್ದವು `vocab_size`, ನಮ್ಮ ಪಠ್ಯದ ಕ್ರಮದ ನಿಜವಾದ ಉದ್ದವನ್ನು ಗಮನದಲ್ಲಿರಿಸದೆ. ನಾವು ಪದದ ಎಂಬೆಡಿಂಗ್‌ಗಳಿಗೆ ಬರುವಾದಾಗ, ಪ್ರತಿ ಪಠ್ಯ ಮಾದರಿಯಲ್ಲಿ ಬದಲಾಗುವ ಪದಗಳ ಸಂಖ್ಯೆ ಇರುತ್ತದೆ, ಮತ್ತು ಆ ಮಾದರಿಗಳನ್ನು ಮಿನಿಬ್ಯಾಚ್‌ಗಳಲ್ಲಿ ಸೇರಿಸುವಾಗ ನಾವು ಕೆಲವು ಪ್ಯಾಡಿಂಗ್ ಅನ್ವಯಿಸಬೇಕಾಗುತ್ತದೆ.

ಇದು ಡೇಟಾಸೋರ್ಸ್ ಗೆ `collate_fn` ಕಾರ್ಯವನ್ನು ನೀಡುವ ಅದೇ ತಂತ್ರವನ್ನು ಬಳಸಿ ಮಾಡಬಹುದು:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### ಎम्बೆಡ್ಡಿಂಗ್ ವರ್ಗವರ್ಗಿಕರಣವನ್ನು ತರಬೇತುಗೊಳಿಸುವುದು

ಈಗ ನಾವು ಸರಿಯಾದ ಡಾಟಲೋಡರ್ ಅನ್ನು ನಿರ்ணಯಿಸಿದ್ದೇವೆ, ನಮ್ಮ ಹಿಂದಿನ ಘಟಕದಲ್ಲಿ ನಿರ್ಣಯಿಸಿರುವ ತರಬೇತಿ ಕಾರ್ಯವನ್ನು ಬಳಸಿಕೊಂಡು ನಾವು ಮಾದರಿಯನ್ನು ತರಬೇತು ಮಾಡಬಹುದು:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **ಗಮನಿಸಿ**: ಸಮಯದ ಕಾರಣಕ್ಕಾಗಿ ನಾವು ಇಲ್ಲಿ ಕೇವಲ 25k ದಾಖಲೆಗಳಿಗೆ ಮಾತ್ರ ತರಬೇತಿ ನೀಡ್ತಿದ್ದೇವೆ (ಒಂದು ಪೂರ್ಣ ಯುಗಕ್ಕಿಂತ ಕಡಿಮೆ), ಆದರೆ ನೀವು ತರಬೇತಿ ಮುಂದುವರಿಸಬಹುದು, ಹಲವು ಯುಗಗಳಿಗೆ ತರಬೇತಿ ನೀಡಲು ಫಂಕ್ಷನ್ ಬರೆಯಬಹುದು, ಮತ್ತು ಉನ್ನತ ನಿಖರತೆ ಸಾಧಿಸಲು ಕಲಿಕಾ ದರ ಪ್ಯಾರಾಮೀಟರ್‌ನೊಂದಿಗೆ ಪ್ರಯೋಗ ಮಾಡಬಹುದು. ನೀವು ಸుమಾರು 90% ನಿಖರತೆಯನ್ನು ತಲುಪಬಹುದು.


### ಎಂಬರ್‌ಡಿಂಗ್‌ಬ್ಯಾಗ್ ಪದರ ಮತ್ತು ಚರಮಾಳದ ಅನುಕ್ರಮ ಅಳೆಯು ತೋರಿಕೆ

ಹಿಂದಿನ ರಚನೆಯಲ್ಲಿ, ನಾವು ಎಲ್ಲ ಚರಮಾಳಗಳನ್ನು ಒಂದೇ ಉದ್ದಕ್ಕೆ ಪ್ಯಾಡ್ ಮಾಡಬೇಕಾಗಿತ್ತು, ώστε ಆವುಗಳನ್ನು ಒಂದು ಮಿನಿಬ್ಯಾಚ್‌ಗೆ ಹೊಂದಿಸಲು. ಇದು ಚರಮಾಳದ ಬದಲಾಗುವ ಉದ್ದದ ಪ್ರತಿನಿಧಿಸುವ ಅತ್ಯಂತ ಪರಿಣಾಮಕಾರಿ ವಿಧಾನವಲ್ಲ - ಮತ್ತೊಂದು ವಿಧಾನವೆಂದರೆ **ಆಫ್‌ಸೆಟ್** ವೆಕ್ಟರ್ ಉಪಯೋಗಿಸುವುದು, ಇದು ಒಂದು ದೊಡ್ಡ ವೆಕ್ಟರ್‌ನಲ್ಲಿ ಸಂರಕ್ಷಿಸಲಾದ ಎಲ್ಲಾ ಚರಮಾಳಗಳ ಆಫ್‌ಸೆಟ್‌ಗಳನ್ನು ಇಡುತ್ತದೆ.

![ಆಫ್‌ಸೆಟ್ ಚರಮಾಳದ ಪ್ರತಿನಿಧಿಸುವಿಕೆಯ ಚಿತ್ರ](../../../../../translated_images/kn/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **ಗಮನಿಸಿ**: ಮೇಲಿನ ಚಿತ್ರದಲ್ಲಿ, ನಾವು ಅಕ್ಷರಗಳ ಒಂದು ಚರಮಾಳ ತೋರಿಸಿದ್ದೇವೆ, ಆದರೆ ನಮ್ಮ ಉದಾಹರಣೆಯಲ್ಲಿ ನಾವು ಪದಗಳ ಚರಮಾಳಗಳೊಂದಿಗೆ ಕೆಲಸ ಮಾಡುತ್ತಿದ್ದೇವೆ. ಆದಾಗ್ಯೂ, ಆಫ್‌ಸೆಟ್ ವೆಕ್ಟರ್‌ನೊಂದಿಗೆ ಚರಮಾಳಗಳನ್ನು ಪ್ರತಿನಿಧಿಸುವ ಸಾಮಾನ್ಯ ತತ್ವ ಏನು ಎಂದಾನೇ ಉಳಿದಿದೆ.

ಆಫ್‌ಸೆಟ್ ಪ್ರತಿನಿಧಿಸುವಿಕೆಯೊಂದಿಗೆ ಕೆಲಸ ಮಾಡಲು, ನಾವು [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) ಪದರವನ್ನು ಉಪಯೋಗಿಸುತ್ತೇವೆ. ಇದು `Embedding` ಹೋಲುತ್ತದೆ, ಆದರೆ ಇದು ವಿಷಯ ವೆಕ್ಟರ್ ಮತ್ತು ಆಫ್‌ಸೆಟ್ ವೆಕ್ಟರ್ ಅನ್ನು ಇನ್ಪುಟ್ ಆಗಿ ತೆಗೆದುಕೊಳ್ಳುತ್ತದೆ, ಮತ್ತು ಇದು ಸರಾಸರಿ ಪದರವನ್ನು ಕೂಡ ಒಳಗೊಂಡಿದೆ, ಅದು `mean`, `sum` ಅಥವಾ `max` ಆಗಿರಬಹುದು.

ಇಲ್ಲಿದೆ `EmbeddingBag` ಉಪಯೋಗಿಸುವ ಪರಿಷ್ಕೃತ ಜಾಲ:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

ತರಬೇತಿಗೆ ಡೇಟಾಸೆಟ್ ಅನ್ನು ತಯಾರಿಸಲು, ನಾವು ಆಫ್‌ಸೆಟ್ ವೆಕ್ಟರ್ ಅನ್ನು ತಯಾರಿಸುವ ಪರಿವರ್ತನಾ ಫಂಕ್ಷನ್ ಅನ್ನು ಒದಗಿಸಬೇಕು:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

ಗಮನಿಸಿ, ಹಿಂದೆ ನೀಡಲಾದ ಎಲ್ಲಾ ಉದಾಹರಣೆಗಳಿಗಿಂತ ಬೇರೆಯಾಗಿ, ನಮ್ಮ ಜಾಲವು ಈಗ ಎರಡು ನಿಯಮಾಂಕಗಳನ್ನು ಸ್ವೀಕರಿಸುತ್ತದೆ: ಡೇಟಾ ವ್ಯಕ್ತರ ಮತ್ತು ಆಫ್ಸೆಟ್ ವ್ಯಕ್ತರ, ಇವು ವಿಭಿನ್ನ ಗಾತ್ರಗಳಿವೆ. ಹೌದೇ ರೀತಿಯಲ್ಲಿ, ನಮ್ಮ ಡೇಟಾ ಲೋಡರ್ ಕೂಡ 2 ಬದಲು 3 ಮೌಲ್ಯಗಳನ್ನು ಒದಗಿಸುತ್ತದೆ: ಪಠ್ಯ ಮತ್ತು ಆಫ್ಸೆಟ್ ವ್ಯಕ್ತರಗಳು ಯಾವುದೂ ವೈಶಿಷ್ಟ್ಯಗಳನ್ನಾಗಿ ಒದಗಿಸಲ್ಪಡುತ್ತವೆ. ಆದ್ದರಿಂದ, ಈ ವಿಷಯಕ್ಕಾಗಿ ನಾವು ನಮ್ಮ ತರಬೇತಿ ಕಾರ್ಯವನ್ನು ಸ್ವಲ್ಪ ಬದಲಾಯಿಸಬೇಕಾಗುತ್ತದೆ:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## ಅರ್ಥಾಂಶ ಎम्बೆಡ್ಡಿಂಗ್ಸ್: Word2Vec

ನಮ್ಮ ಹಿಂದಿನ ಉದಾಹರಣೆಯಲ್ಲಿ, ಮಾದರಿ ಎम्बೆಡ್ಡಿಂಗ್ ಲೇಯರ್ ಪದಗಳನ್ನು ವೆಕ್ಟರ್ ಪ್ರತಿನಿಧಿಸುವಿಕೆಗೆ ಕಲಿತು, ಆದರೆ, ಈ ಪ್ರತಿನಿಧಿಸಿರುವಿಕೆ ಹೆಚ್ಚು ಅರ್ಥಪೂರ್ಣವಾಗಿರಲಿಲ್ಲ. ಅಂತಹ ವೆಕ್ಟರ್ ಪ್ರತಿನಿಧಿಸುವಿಕೆಯಾದರೆ ಚೆನ್ನಾಗಿತ್ತು, ಅಂದರೆ ಸಮಾನಾರ್ಥಕ ಪದಗಳು ಅಥವಾ ಸಮಾನ ಪದಗಳು ಒಟ್ಟಿಗೆ ಇರುವ ವೆಕ್ಟರ್ಗಳಿಗೆ ಸಮೀಪವಾಗಿರಬೇಕು ಎಂದಾದರೆ ಆವೆಕ್ಟರ್ ದೂರದ (ಉದಾ. ಯೂಕ್ಲಿಡಿಯನ್ ದೂರ) ದೃಷ್ಟಿಯಿಂದ.

ಅದನ್ನು ಮಾಡಲು, ನಾವು ನಮ್ಮ ಎಂಬೆಡ್ಡಿಂಗ್ ಮಾದರಿಯನ್ನು ಒಂದು ನಿರ್ದಿಷ್ಟ ವಿಧದಲ್ಲಿ ದೊಡ್ಡ ಪಠ್ಯ ಸಂಗ್ರಹದಲ್ಲಿ ಪೂರ್ವ-ಪ್ರಶಿಕ್ಷಣ ಮಾಡಬೇಕಾಗುತ್ತದೆ. ಅರ್ಥಾಂಶ ಎम्बೆಡ್ಡಿಂಗ್ಸ್ ತರಬೇತಿಗೆ ಮೊದಲ ವಿಧಾನಗಳಲ್ಲಿ ಒಂದಾಗಿ [Word2Vec](https://en.wikipedia.org/wiki/Word2vec) ಹೆಸರು ಪಡೆದಿದೆ. ಇದು ಎರಡು ಮುಖ್ಯ ವಿನ್ಯಾಸಗಳ ಮೇಲೆ ಆಧರಿತವಾಗಿದೆ, ಅವು ಪದಗಳ ವಿವರಣಾತ್ಮಕ ಪ್ರತಿನಿಧಿಸಿಕೆಗೆ ಉಪಯೋಗಿಸುತ್ತವೆ:

 - **ಸತತ ಬ್ಯಾಗ್-ಆಫ್-ವರ್ಡ್ಸ್** (CBoW) — ಈ ವಿನ್ಯಾಸದಲ್ಲಿ, ಮಾದರಿಯನ್ನು ಸುತ್ತುವರೆದ ಪರಿಶುದ್ಧ ಪದಗಳಿಂದ ಒಂದು ಪದವನ್ನು ಊಹಿಸಲು ತರಬೇತಿಮಾಡಲಾಗುತ್ತದೆ. ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$ ಕೊಡಲಾಗುವಾಗ, ಮಾದರಿಯ ಗುರಿ $W_0$ ಅನ್ನು $(W_{-2},W_{-1},W_1,W_2)$ ನಿಂದ ಊಹಿಸುವುದು.
 - **ಸತತ ಸ್ಕಿಪ್-ಗ್ರಾಂ** CBoW ಗೆ ವಿರುದ್ಧವಾಗಿದೆ. ಮಾದರಿ ಸುತ್ತಲೂ ಇರುವ ಪರಿಧಿಯ ಪದಗಳನ್ನು ಉಪಯೋಗಿಸಿ ಪ್ರಸ್ತುತ ಪದವನ್ನು ಊಹಿಸುತ್ತದೆ.

CBoW ವೇಗವಾಗಿರುತ್ತದೆ, ಆದರೆ ಸ್ಕಿಪ್-ಗ್ರಾಂ ನಿಧಾನವಾಗಿದ್ದರೂ, ಅಪರೂಪದ ಪದಗಳನ್ನು ಉತ್ತಮವಾಗಿ ಪ್ರತಿನಿಧಿಸುತ್ತದೆ.

![ಪದಗಳನ್ನು ವೆಕ್ಟರ್ಗಳಿಗೆ ಪರಿವರ್ತಿಸಲು CBoW ಮತ್ತು ಸ್ಕಿಪ್-ಗ್ರಾಂ ಗಳನ್ನು ತೋರಿಸುವ ಚಿತ್ರ.](../../../../../translated_images/kn/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Google News ಡೇಟಾಸೇಟ್ನಲ್ಲಿ ಪೂರ್ವ-ಪ್ರಶಿಕ್ಷಿತವಾದ word2vec ಎम्बೆಡ್ಡಿಂಗ್ ಜೊತೆಪ್ರಯೋಗ ಮಾಡಲು, ನಾವು **gensim** ಗ್ರಂಥಾಲಯವನ್ನು ಬಳಸಬಹುದು. ಕೆಳಗಿನವು 'neural' ಗೆ ಅತ್ಯಂತ ಸಾದೃಶ್ಯ ಹೊಂದಿರುವ ಪದಗಳು.

> **ಸೂಚನೆ:** ನೀವು ಮೊದಲ ಬಾರಿ ಪದ ವೆಕ್ಟರ್ಗಳನ್ನು ರಚಿಸುವಾಗ, ಅವುಗಳನ್ನು ಡೌನ್‌ಲೋಡ್ ಮಾಡಿಕೊಳ್ಳಲು ಸ್ವಲ್ಪ ಸಮಯ ಬೇಕಾಗಬಹುದು!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


ವರ್ಗೀಕರಣ ಮಾದರಿಯನ್ನು ತರಬೇತಿಗೆ ಬಳಸಲು, ನಾವು ಪದದಿಂದ ವೆಕ್ಟರ್ ನೇರೀಕರಣಗಳನ್ನು ಗಣನೆ ಮಾಡಬಹುದು (ಸ್ಪಷ್ಟತೆಗೆ ವೆಕ್ಟರ್‌ನ ಮೊದಲು 20 ಘಟಕಗಳನ್ನು ಮಾತ್ರ ತೋರಿಸಲಾಗುತ್ತದೆ):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

ಅರ್ಥಸಂಭಂಧಿತ ಎಂಬೆಡ್ಡಿಂಗ್‌ಗಳ ಬಗ್ಗೆ ಅದ್ಭುತ ಸಂಗತಿ ಏನೆಂದರೆ ನೀವು ಸೇಮ್ಯಾಂಟಿಕ್ಸ್ ಬದಲಾಯಿಸಲು ವೆಕ್ಟರ್ ಎನ್‌ಕೋಡಿಂಗ್ ಅನ್ನು ಮಾನಿಪ್ಯುಲೇಟ್ ಮಾಡಬಹುದು. ಉದಾಹರಣೆಗೆ, ನಾವು *ರಾಜಿ* ಮತ್ತು *ಹೆಣ್ಣು* ಎಂಬ ಪದಗಳಿಗೆ ಸಾಧ್ಯವಾದಷ್ಟು ಹತ್ತಿರವಾಗಿರುವ ಮತ್ತು *ಪುರುಷ* ಎಂಬ ಪದದಿಂದ ಸಿಬ್ಬಂದಿ ದೂರವಾಗಿರುವ ಪದವನ್ನು ಹುಡುಕಲು ಕೇಳಬಹುದು:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

CBoW ಮತ್ತು Skip-Grams ಎರಡೂ “ಭವಿಷ್ಯವಾಣಿನ” ಎಂಬರ್‍ಡಿಂಗ್ ಗಳು, ಅವುಗಳು ಸ್ಥಳೀಯ ಸನ್ನಿವೇಶಗಳನ್ನು ಮಾತ್ರ ಪರಿಗಣಿಸುವುದರಲ್ಲಿ ಇದ್ದವೆ. Word2Vec ಜಾಗತಿಕ ಸನ್ನಿವೇಶದ ಲಾಭವನ್ನು ತೆಗೆದುಕೊಳ್ಳುವುದಿಲ್ಲ. 

**FastText**, Word2Vec ಮೇಲೆ ಆಧರಿಸಿದೆ, ಪ್ರತಿ ಪದ ಮತ್ತು ಆ ಪದದಲ್ಲಿ ಕಂಡುಬರುವ ಅಕ್ಷರ n-ಗ್ರಾಂಗಳನ್ನು.vector ಪ್ರತಿನಿಧಿಗಳನ್ನು ಕಲಿಯುತ್ತದೆ. ಪ್ರತಿನಿಧಿಗಳ ಮೌಲ್ಯಗಳನ್ನು ಪ್ರತಿ ತರಬೇತಿ ಹಂತದಲ್ಲಿ ಒದೆಯಾಗಿ ಒಂದು ವೆಕ್ಟರ್ ಆಗಿ ಸರಾಸರಿ ಮಾಡಲಾಗುತ್ತದೆ. ಇದರಿಂದ ಪೂರ್ವ-ತರಬೇತಿಗೆ ಹೆಚ್ಚಿನ ಗಣನೆ ಸೇರಿದ್ದರೂ, ಪದದ ಎಂಬರ್‍ಡಿಂಗ್ಗಳು ಉಪ-ಪದ ಮಾಹಿತಿಯನ್ನು ಎನ್‌ಕೋಡ್ ಮಾಡಬಹುದು. 

ಇನ್ನೊಂದು ವಿಧಾನ, **GloVe**, ಸಹ-ಸಂಘಟನೆ ಮಾಟ್ರಿಕ್ಸ್ ಅನ್ನುವ ಆಲೋಚನೆಯನ್ನು ಬಳಸುತ್ತದೆ, ಸಹ-ಸಂಘಟನೆ ಮಾಟ್ರಿಕ್ಸ್ ಅನ್ನು ವಿವರಣಾತ್ಮಕ ಹಾಗೂ ರೇಖೆಯಲ್ಲದ ಪದ ವೆಕ್ಟರ್ಗಳಾಗಿ ವಿಭಜಿಸಲು ನಯೂರಾಲ್ ವಿಧಾನಗಳನ್ನು ಬಳಸುತ್ತದೆ.

ನೀವು ಉದಾಹರಣೆಯೊಂದಿಗೆ ಆಡಬಹುದು, ಎಂಬರ್‍ಡಿಂಗ್ಸ್ ಅನ್ನು FastText ಮತ್ತು GloVe ಗೆ ಬದಲಾಯಿಸುವ ಮೂಲಕ, ಏಕೆಂದರೆ gensim ಹಲವಾರು ವಿಭಿನ್ನ ಪದ ಎಂಬರ್‍ಡಿಂಗ್ ಮಾದರಿಗಳನ್ನು ಬೆಂಬಲಿಸುತ್ತದೆ.


## ಪೈಟಾರ್ಚ್‌ನಲ್ಲಿ ಪೂರ್ವ-ಪ್ರಶಿಕ್ಷಿತ ಎಂಬೇಡಿಂಗ್‌ಗಳನ್ನು ಬಳಸುವುದು

ಮೇಲಿನ ಉದಾಹರಣೆಯನ್ನು ನಾವು ನಮ್ಮ ಎಂಬೇಡಿಂಗ್ ಲೇಯರ್‌ನಲ್ಲಿ ಸೆಮಾಂತಿಕ ಎಂಬೇಡಿಂಗ್‌ಗಳೊಂದಿಗೆ, ಉದಾಹರಣೆಗೆ Word2Vec, ಮೆಟ್ರಿಕ್ಸ್ ಅನ್ನು ಪೂರ್ವ-ಭರ್ತಿ ಮಾಡಲು ಪರಿಷ್ಕರಿಸಬಹುದು. ಪೂರ್ವ-ಪ್ರಶಿಕ್ಷಿತ ಎಂಬೇಡಿಂಗ್ ಮತ್ತು ನಮ್ಮ ಪಠ್ಯ ನಿಘಂಟಿನ ಅವಿಭಾಜ್ಯತೆಗಳು ಹೊಂದಿಕೆಯಾಗುವ ಸಾಧ್ಯತೆ ಕಡಿಮೆ ಎಂದು ಗಮನದಲ್ಲಿಟ್ಟುಕೊಳ್ಳಬೇಕಾಗುತ್ತದೆ, ಆದ್ದರಿಂದ ನಾವು ಕಳೆದುಹೋದ ಪದಗಳಿಗೆ ತೂಕಗಳನ್ನು ಸಾಂಖ್ಯಿಕ ಮೌಲ್ಯಗಳಿಂದ ಪ್ರಾರಂಭಿಸೋಣ:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


ಈಗ ನಮ್ಮ ಮಾದರಿಯನ್ನು ತರಬೇತಿಮಾಡೋಣ. ಮಾದರಿಯನ್ನು ತರಬೇತಿಮಾಡಲು ತೆಗೆದುಕೊಳ್ಳುವ ಸಮಯವು ಹಿಂದಿನ ಉದಾಹರಣೆಯ ಪೈಕಿ ಬಹಳಷ್ಟು ಹೆಚ್ಚಾಗಿದೆ ಎಂದು ಗಮನಿಸಿ, ಇದಕ್ಕೆ ಕಾರಣ embedding ಲೇಯರ್ ಗಾತ್ರದ ಹೆಚ್ಚಳ ಹಾಗೂ ಅಷ್ಟು ಹೆಚ್ಚು ಪ್ಯಾರಾಮೀಟರ್‌ಗಳಿರುತ್ತವೆ. ಇದರಿಂದಾಗಿಯೇ, ನಾವು ಉಳಿದುಕೊಳ್ಳುವಿಕೆಯನ್ನು ತಪ್ಪಿಸಲು ಬೇಕಾದರೆ, ನಮ್ಮ ಮಾದರಿಯನ್ನು ಹೆಚ್ಚು ಉದಾಹರಣೆಗಳಲ್ಲಿ ತರಬೇತಿಮಾಡಬೇಕಾಗಬಹುದು.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

ನಮ್ಮ ಪ್ರಕರಣದಲ್ಲಿ ನಾವು ಹೆಚ್ಚಿದ ಶುದ್ಧತೆಗೆ ಕಂಡುಬರಲಿಲ್ಲ, ಇದು ಬಹುಶಃ ವಿಭಿನ್ನ ಶಬ್ದಕೋಶಗಳಿರಲಿರುವ ಸಾಧ್ಯತೆ ಇದೆಯೆಂದು ಕಂಡುಬರುತ್ತದೆ.
ವಿಭಿನ್ನ ಶಬ್ದಕೋಶಗಳ ಸಮಸ್ಯೆಯನ್ನು ಮೀರುವುದುಗಾಗಿ, ನಾವು ಕೆಳಗಿನ ಪರಿಹಾರಗಳಲ್ಲಿ ಒಂದನ್ನು ಬಳಸಬಹುದು:
* ನಮ್ಮ ಶಬ್ದಕೋಶದಲ್ಲಿ word2vec ಮಾದರಿಯನ್ನು ಮರುಪ್ರಶಿಕ್ಷಣ ಮಾಡು
* ಪೂರ್ವ-ಪ್ರಶಿಕ್ಷಿತ word2vec ಮಾದರಿಯಿಂದ ಶಬ್ದಕೋಶವನ್ನು ಬಳಸಿಕೊಂಡು ನಮ್ಮ ದತ್ತಸಂಚಯವನ್ನು ಲೋಡ್ ಮಾಡು. ದತ್ತಸಂಚಯವನ್ನು ಲೋಡ್ ಮಾಡುವಾಗ ಬಳಸಲಾದ ಶಬ್ದಕೋಶವನ್ನು ನಿರ್ದಿಷ್ಟಪಡಿಸಬಹುದು.

ಎರಡನೇ ವಿಧಾನ ಸುಲಭವೆಂದು ತೋರುತ್ತದೆ, ವಿಶೇಷವಾಗಿ ಏಕೆಂದರೆ PyTorch `torchtext` ತಂತ್ರಾಂಶದಲ್ಲಿ ಎम्बೆಡ್ಡಿಂಗ್‌ಗಳಿಗೆ ಒಳಗೊಂಡ ಬೆಂಬಲವಿದೆ. ಉದಾಹರಣೆಗೆ, ನಾವು GloVe ಆಧಾರಿತ ಶಬ್ದಕೋಶವನ್ನು ಕೆಳಗಿನ ರೀತಿಯಲ್ಲಿ ಉಲ್ಲೇಖಿಸಬಹುದು:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary has the following basic operations:
* `vocab.stoi` dictionary allows us to convert word into its dictionary index
* `vocab.itos` does the opposite - converts number into word
* `vocab.vectors` is the array of embedding vectors, so to get the embedding of a word `s` we need to use `vocab.vectors[vocab.stoi[s]]`

Here is the example of manipulating embeddings to demonstrate the equation **kind-man+woman = queen** (I had to tweak the coefficient a bit to make it work):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

ಆ ಸಂಯೋಜನಗಳನ್ನು ಬಳಸಿ ವರ್ಗೀಕರಣಕಾರರನ್ನು ತರಬೇತಿಮಾಡಲು, ಮೊದಲು ನಾವು ನಮ್ಮ ಡೇಟಾಸೆಟ್ ಅನ್ನು GloVe ಪದಕೋಶ ಬಳಸಿ ಎನ್‌ಕೋಡ್ ಮಾಡಬೇಕಾಗಿದೆ:  


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

ಮೇಲಿನಂತೆ ನಾವು ನೋಡಿದಂತೆ, ಎಲ್ಲಾ ವೆಕ್ಟರ್ ಎम्बೆಡ್ಡಿಂಗ್ಗಳನ್ನು `vocab.vectors` ಮ್ಯಾಟ್ರಿಕ್ಸ್‌ನಲ್ಲಿ ಸಂಗ್ರಹಿಸಲಾಗಿದೆ. ಇದರಿಂದ ಆ ಎಂಬೆಡ್ಡಿಂಗ್ ಲೇಯರ್‌ನ ತೂಕಗಳಿಗೆ ಆ ತೂಕಗಳನ್ನು ಸರಳ ನಕಲಿಸುವ ಮೂಲಕ ತುಂಬಾ ಸುಲಭವಾಗಿ ಲೋಡ್ ಮಾಡಬಹುದು:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

ಈಗ ನಾವು ನಮ್ಮ ಮಾದರಿಯನ್ನು ತರಬೇತಿ ಮಾಡೋಣ ಮತ್ತು ಉತ್ತಮ ಫಲಿತಾಂಶಗಳು ಸಿಗುತ್ತವೆಯೇ ನೋಡೋಣ:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

ನಾವು ಗಮನಾರ್ಹವಾದ ಶುದ್ಧತೆಯ ಏರಿಕೆಯನ್ನು ಕಾಣದಿರುವ ಕಾರಣಗಳಲ್ಲಿ ಒಂದಾಗಿದೆ ನಮ್ಮ ಡೇಟಾಸೆಟ್‌ನ ಕೆಲವು ಪದಗಳು ಪೂರ್ವ-ಪ್ರಶಿಕ್ಷಿತ GloVe ಪದಕೋಶದಲ್ಲಿ ಇಲ್ಲದಿರುವುದು, ಆದ್ದರಿಂದ ಅವುಗಳನ್ನು ಪ್ರಧಾನವಾಗಿ ನಿರ್ಲಕ್ಷಿಸಲಾಗಿದೆ. ಈ ವಾಸ್ತವವನ್ನು ಗೆಲ್ಲಲು, ನಾವು ನಮ್ಮ ಡೇಟಾಸೆಟ್‌ನಲ್ಲಿ ನಮ್ಮದೇ embeddings ಅನ್ನು ತರಬೇತಿ ಮಾಡಬಹುದು. 


## ಸಾಂದರ್ಭಿಕ ವರನಿಗಳು

Word2Vec ಹಾಗು ಇಂತಹ ಸಾಂಪ್ರದಾಯಿಕ ಪೂರ್ವಭಾವಿ ವರನ ಪ್ರತಿನಿಧಿಗಳ ಪ್ರಮುಖ ನಿಯತೆಯೊಂದು ಪದಾರ್ಥದ ಅರ್ಥ ನಿರ್ಣಯದ ಸಮಸ್ಯೆಯಾಗಿದೆ. ಪೂರ್ವಭಾವಿ ವರನಗಳು ಸಾಂದರ್ಭಿಕ ಪದಗಳ ಅರ್ಥವನ್ನು ಕೆಲವು ಭಾಗಿಸಬಹುದು, ಆದರೆ ಪದದ ಪ್ರತಿ ಸಾಧ್ಯ ಅರ್ಥವನ್ನು ಒಂದೇ ವರನದಲ್ಲಿ ಹೊಂದಿಸುತ್ತವೆ. ಇದು ತಳಹದಿ ಮಾದರಿಗಳಲ್ಲಿ ಸಮಸ್ಯೆಗಳನುಂಟುಮಾಡಬಹುದು, ಏಕೆಂದರೆ 'play' ಎಂಬ ಪದಕ್ಕೂ ವಿವಿಧ ಸಾಂದರ್ಭಿಕ ಅರ್ಥಗಳು ಇವೆ.

ಉದಾಹರಣೆಗೆ 'play' ಎಂಬ ಪದ ಎರಡು ವಿಭಿನ್ನ ವಾಕ್ಯಗಳಲ್ಲಿ ತುಂಬಾ ವಿಭಿನ್ನ ಅರ್ಥವನ್ನು ಹೊಂದಿದೆ:
- ನಾನು ನಾಟಕಶಾಲೆಯಲ್ಲಿ ಒಂದು **ನಾಟಕ** ನೋಡಲು ಹೋದೆನು.
- ಜಾನ್ ತನ್ನ ಸ್ನೇಹಿತರೊಂದಿಗೆ **ಆಟವಾಡಲು** ಇಚ್ಛಿಸುತ್ತಿದ್ದಾನೆ.

ಮೇಲಿನ ಪೂರ್ವಭಾವಿ ವರನಗಳು 'play' ಎಂಬ ಪದದ ಈ ಎರಡೂ ಅರ್ಥಗಳನ್ನು ಒಂದೇ ವರನದಲ್ಲಿ ಪ್ರತಿನಿಧಿಸುತ್ತವೆ. ಈ ನಿಯತೆಯನ್ನು ದಾಟಲು, ನಾವು **ಭಾಷಾ ಮಾದರಿಗಳ** ಆಧಾರದ ಮೇಲೆ ವರನಗಳನ್ನು ನಿರ್ಮಿಸಬೇಕು, ಅವು ದೊಡ್ಡ ಪಠ್ಯ ಸಂಗ್ರಹದಲ್ಲಿ ತರಬೇತುಗೊಂಡಿದ್ದು ಪದಗಳನ್ನು ವಿವಿಧ ಸಾಂದರ್ಭಗಳಲ್ಲಿ ಹೇಗೆ ಸೇರಿಸಬಹುದು ಎಂದು *ಗುರ್ತಿಸುತ್ತವೆ.* ಸಾಂದರ್ಭಿಕ ವರನಗಳ ಬಗ್ಗೆ ಚರ್ಚಿಸುವುದು ಈ ಪಾಠದ ವ್ಯಾಪ್ತಿಗೆ ಹೊರತರಲು ಇರುತ್ತದೆ, ಆದರೆ ಭಾಷಾ ಮಾದರಿಗಳ ಬಗ್ಗೆ ಮುಂದಿನ ಘಟಕದಲ್ಲಿ ಮಾತನಾಡುವಾಗ ನಾವು ಅದಕ್ಕೆ ಮತ್ತೆ ಬರುವೇವೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
